# 📊 Visualization — B1 Descriptive và B2 Distribution

Notebook đọc dữ liệu feature từ `data/processed/commits_features.csv`. Tất cả hình được lưu vào `reports/figures/` để chèn vào báo cáo. Ngày giờ được phân tích theo UTC.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # chạy được cả khi không có GUI/Tkinter
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
DATA_PATH = Path('../data/processed/commits_features.csv')
FIG_DIR = Path('../reports/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH, low_memory=False)
df['commit_date'] = pd.to_datetime(df['commit_date'], utc=True, errors='coerce')
if 'change_commit_date' in df: df['change_commit_date'] = pd.to_datetime(df['change_commit_date'], utc=True, errors='coerce')
print(f'✅ Đã tải {DATA_PATH}: {len(df):,} dòng × {df.shape[1]} cột')

def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout(); plt.savefig(path, dpi=160, bbox_inches='tight'); print(f'Đã lưu {path}')
    plt.show(); plt.close()

✅ Đã tải ../data/processed/commits_features.csv: 21,498 dòng × 23 cột


## B1 — Descriptive Visualization

In [2]:
# B1.1 — Số lượng theo năm
year_counts = df.groupby('year').size().sort_index()
year_labels = [f'{y} (T1–T3)' if y == 2026 else str(y) for y in year_counts.index]  # dữ liệu 2026 chỉ có quý 1
plt.figure(figsize=(8, 4.5)); ax = sns.barplot(x=year_labels, y=year_counts.values, color='#16a34a')
ax.set(title='Số khối code có comment AI theo năm', xlabel='Năm', ylabel='Số khối code')
for container in ax.containers: ax.bar_label(container, fmt='%.0f')
savefig('B1_01_commits_by_year.png')

# B1.2 — Tỷ lệ ngôn ngữ
lang = df['language'].value_counts()
plt.figure(figsize=(5.5, 5.5)); plt.pie(lang.values, labels=[f'{k}: {v:,}' for k,v in lang.items()], autopct='%.1f%%', startangle=90, colors=['#3776AB', '#F7DF1E'])
plt.title('Tỷ lệ ngôn ngữ lập trình'); savefig('B1_02_language_share.png')

# B1.3 — Heatmap giờ × thứ
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heat = pd.crosstab(df['day_of_week'], df['hour']).reindex(index=day_order, columns=range(24), fill_value=0)
plt.figure(figsize=(16, 4.5)); sns.heatmap(heat, cmap='turbo', linewidths=.3); plt.title('Mật độ AI commit theo giờ và thứ (UTC)'); plt.xlabel('Giờ'); plt.ylabel('Ngày trong tuần'); savefig('B1_03_hour_day_heatmap.png')

# B1.4 — Tăng trưởng theo tháng
monthly = df.groupby('year_month').size().sort_index()
plt.figure(figsize=(13, 4.5)); plt.plot(monthly.index, monthly.values, marker='o', linewidth=1.8, color='#2563eb'); plt.xticks(rotation=60); plt.title('Tăng trưởng AI commit theo tháng'); plt.xlabel('Tháng'); plt.ylabel('Số khối code'); savefig('B1_04_monthly_growth.png')

# B1.5 — Tỷ lệ has_recorded_change (donut)
modified = df['has_recorded_change'].value_counts().reindex([False, True], fill_value=0)
plt.figure(figsize=(5.5, 5.5)); plt.pie(modified.values, labels=['Không ghi nhận thay đổi','Có ghi nhận thay đổi'], autopct='%.1f%%', startangle=90, colors=['#22c55e','#ef4444'], wedgeprops={'width':.42, 'edgecolor':'white'}); plt.title('Khối code có ghi nhận lần thay đổi đầu tiên'); savefig('B1_05_modified_donut.png')

# B1.6 — Top 15 repo theo số interaction
top_repo = df['repo'].value_counts().head(15).sort_values()
plt.figure(figsize=(9, 6)); sns.barplot(x=top_repo.values, y=top_repo.index, color='#7c3aed'); plt.title('Top 15 repository có nhiều AI interaction nhất'); plt.xlabel('Số khối code'); plt.ylabel('Repository'); savefig('B1_06_top_repositories.png')

# B1.7 — Công cụ AI
tool = df['ai_tool'].value_counts().sort_values(ascending=False)
plt.figure(figsize=(9, 4.8)); sns.barplot(x=tool.values, y=tool.index, color='#0891b2'); plt.title('Phân bố công cụ AI (heuristic)'); plt.xlabel('Số khối code'); plt.ylabel('AI tool'); savefig('B1_07_ai_tool.png')

# B1.8 — Mức đóng góp / ai_verb
verb = df['ai_verb'].value_counts().sort_values(ascending=False)
plt.figure(figsize=(8, 4.5)); sns.barplot(x=verb.values, y=verb.index, color='#f97316'); plt.title('Mức đóng góp AI (heuristic)'); plt.xlabel('Số khối code'); plt.ylabel('ai_verb'); savefig('B1_08_ai_verb.png')

# B1.9 — Python vs JS theo tháng (stacked area)
lang_month = pd.crosstab(df['year_month'], df['language']).sort_index()
plt.figure(figsize=(13, 4.8)); plt.stackplot(lang_month.index, [lang_month.get(c, pd.Series(0, index=lang_month.index)) for c in ['python','js']], labels=['python','js'], colors=['#3776AB','#F7DF1E']); plt.xticks(rotation=60); plt.title('Python vs JavaScript theo tháng'); plt.xlabel('Tháng'); plt.ylabel('Số khối code'); plt.legend(loc='upper left'); savefig('B1_09_language_month_area.png')

# B1.12 — Tỷ lệ có ghi nhận thay đổi theo ai_tool (cận dưới của tỷ lệ bị sửa, không đo độ tin cậy vì không có nhóm đối chứng)
tool_rate = df.groupby('ai_tool')['has_recorded_change'].agg(n='size', change_rate='mean').sort_values('change_rate')
ax = tool_rate['change_rate'].mul(100).plot(kind='barh', figsize=(9, 5), color='#16a34a'); ax.set(title='Tỷ lệ có ghi nhận thay đổi theo AI tool', xlabel='%', ylabel='ai_tool'); savefig('B1_12_tool_change_rate.png')
display(tool_rate.sort_values('change_rate', ascending=False).round(3))

Đã lưu ../reports/figures/B1_01_commits_by_year.png
Đã lưu ../reports/figures/B1_02_language_share.png


Đã lưu ../reports/figures/B1_03_hour_day_heatmap.png


Đã lưu ../reports/figures/B1_04_monthly_growth.png
Đã lưu ../reports/figures/B1_05_modified_donut.png


Đã lưu ../reports/figures/B1_06_top_repositories.png


Đã lưu ../reports/figures/B1_07_ai_tool.png


Đã lưu ../reports/figures/B1_08_ai_verb.png


Đã lưu ../reports/figures/B1_09_language_month_area.png


Đã lưu ../reports/figures/B1_12_tool_change_rate.png


,n,change_rate
ai_tool,,
claude,4192,0.425
other_ai,6600,0.350
other_tool,35,0.343
chatgpt_gpt,6437,0.330
gemini_bard,1113,0.318
github_copilot,2702,0.301
llama_meta,419,0.253


## B2 — Distribution Analysis

In [3]:
# Chỉ dùng số ngày hợp lệ (notebook 03 đã chuyển khoảng ≤ 0 thành NaN)
days = df['days_to_modification'].dropna()
# B2.1 — Histogram + KDE
plt.figure(figsize=(9, 4.8)); sns.histplot(days, bins=50, kde=True, color='#2563eb'); plt.title('Phân phối days_to_modification'); plt.xlabel('Số ngày đến lần sửa đầu tiên'); plt.ylabel('Số khối code'); savefig('B2_01_days_hist_kde.png')

# B2.2 — Code lines theo ngôn ngữ (boxplot + violin)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8)); sns.boxplot(data=df, x='language', y='code_lines', ax=axes[0], showfliers=False); axes[0].set_title('Boxplot code_lines'); sns.violinplot(data=df, x='language', y='code_lines', ax=axes[1], cut=0); axes[1].set_title('Violin code_lines'); fig.suptitle('Phân phối kích thước code theo ngôn ngữ'); savefig('B2_02_code_lines_language.png')

# B2.3 — Q-Q plot trên log1p(days), phù hợp với phân phối lệch phải
plt.figure(figsize=(6, 6)); stats.probplot(np.log1p(days), dist='norm', plot=plt); plt.title('Q-Q plot log1p(days_to_modification)'); savefig('B2_03_qq_days.png')

# B2.4 — Correlation matrix
corr_cols = ['year','month','hour','is_weekend','has_recorded_change','days_to_modification','code_lines','code_chars','repo_commit_count']
corr = df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 8)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, square=True); plt.title('Correlation matrix của các numeric features'); savefig('B2_04_correlation_matrix.png')
print(f'✅ B2.1 dùng {len(days):,} dòng days_to_modification hợp lệ')

Đã lưu ../reports/figures/B2_01_days_hist_kde.png


Đã lưu ../reports/figures/B2_02_code_lines_language.png


Đã lưu ../reports/figures/B2_03_qq_days.png


Đã lưu ../reports/figures/B2_04_correlation_matrix.png
✅ B2.1 dùng 5,758 dòng days_to_modification hợp lệ
